# 28 · School access patterns: hotspots, area types, rules, underserved clusters — Part D, step D7 (schools)

**Inputs:** notebook 27's tables (India sub-districts, Kerala local bodies; state-levels with an incomplete UDISE
school list left out: Delhi all levels, DNH & DD all levels, Andhra Pradesh classes 11–12) and notebook 25's Kerala
cells. Units with fewer than 100 children of a level are left out of that level.

1. **Gi\* hotspots** (Getis & Ord 1992; Sedona GeoStats, analytical z): where poor access clusters — hot spots of
   the share of children beyond the distance norm, cold spots of places per child. Neighbours: binary distance band
   between unit centroids (99th percentile of nearest-neighbour distances), self included. Benjamini–Hochberg FDR 5 %.
   Checked against the formula computed directly.
2. **K-Means area types** (Spark MLlib) on India sub-districts: 12 standardised features (per level: % beyond the
   norm, log distance, log places per child); K by silhouette; stability under a second seed.
3. **FP-Growth** (Spark MLlib): which access problems occur together (far from school / few places, per level);
   support, confidence, lift; recounted in pandas.
4. **Underserved clusters, Kerala:** connected 100 m cells with no school of the level within the norm (same as DBSCAN,
   eps 150 m, minPts 1); clusters with ≥ 50 children are the candidates for D7b (where to build).

**Outputs** (`data/lake/gold/`, copied to `FINAL COMPLETE DATASET/results/schools/`): `school_hotspots_*`,
`school_gistar_check.csv`, `school_kmeans_silhouette.csv`, `school_area_types_india_subdistrict.csv`,
`school_area_type_profiles.csv`, `school_fp_rules.csv`, `school_underserved_clusters_kerala.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona, read the D6 tables

In [4]:
import shutil

import numpy as np
import pandas as pd
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.fpm import FPGrowth
from pyspark.sql import functions as F
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.stats import norm
from sedona.spark.stats.hotspot_detection.getis_ord import g_local
from sedona.spark.stats.weighting import add_binary_distance_band_column
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc
import school_capacity as sc

SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
RESULTS = PROJECT / "FINAL COMPLETE DATASET" / "results" / "schools"
LEVELS = list(sc.INDIA_LEVELS)
LEVELS_K = ["lp", "up", "hs", "hss"]
MIN_CHILDREN = 100  # units with fewer children of a level are left out of that level (too few for a stable rate)
spark = get_spark(PROJECT, cores=4, memory="3g")  # small data (thousands of areas); leaves RAM for the rest of the laptop

# D6 results; state-levels whose UDISE 2022 school list is incomplete are left out (notebook 27)
cov = pd.read_csv(GOLD / "school_data_coverage.csv")
FLAG = {lv: set(cov.loc[cov["udise_incomplete"] & cov["level"].eq(lv), "state"]) for lv in LEVELS}
print("left out (UDISE incomplete):", {lv: sorted(s) for lv, s in FLAG.items()})
sub = pd.read_csv(GOLD / "school_india_by_subdistrict.csv").query("subdistrict_code > 0")
lb = pd.read_csv(GOLD / "school_kerala_by_local_body.csv").query("state == 'Kerala' and local_body_code > 0")


def centroids(code_col, codes):
    """Unit centroids in metres (EPSG:7755, India-wide Lambert conformal conic)."""
    u = pc.load_units(SILVER, code_col).set_crs(4326, allow_override=True)
    u = u[u["code"].isin(codes)].to_crs(7755)
    return pd.DataFrame({code_col: u["code"].to_numpy(), "x": u.centroid.x.to_numpy(), "y": u.centroid.y.to_numpy()})


assert sub["subdistrict_code"].is_unique and lb["local_body_code"].is_unique, "a unit has more than one row"
sub = sub.merge(centroids("subdistrict_code", sub["subdistrict_code"]), on="subdistrict_code")
lb = lb.merge(centroids("local_body_code", lb["local_body_code"]), on="local_body_code")
print("India sub-districts:", len(sub), "| Kerala local bodies:", len(lb))

left out (UDISE incomplete): {'primary': ['Delhi', 'Dādra and Nagar Haveli and Damān and Diu'], 'upper_primary': ['Delhi', 'Dādra and Nagar Haveli and Damān and Diu'], 'secondary': ['Delhi', 'Dādra and Nagar Haveli and Damān and Diu'], 'higher_secondary': ['Andhra Pradesh', 'Delhi', 'Dādra and Nagar Haveli and Damān and Diu']}


India sub-districts: 6810 | Kerala local bodies: 1027


## 1. Gi* hotspots

In [5]:
# Gi* hotspots (Getis & Ord 1992; Sedona GeoStats, analytical z). Neighbours: binary distance band between unit
# centroids, self included (Gi*). Band = 99th percentile of the nearest-neighbour distances, so 99 % of units have a
# neighbour; units without one (islands, Ladakh) get no Gi*. Multiple testing: Benjamini-Hochberg FDR 5 %.
from scipy.spatial import cKDTree


def band(df):
    d, _ = cKDTree(df[["x", "y"]].to_numpy()).query(df[["x", "y"]].to_numpy(), k=2)
    return float(np.quantile(d[:, 1], 0.99))


def fdr(p, q=0.05):
    """Benjamini-Hochberg: True where the p-value is significant at false discovery rate q."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    passed = np.zeros(len(p), bool)
    k = np.nonzero(ranked <= q)[0]
    if len(k):
        passed[order[: k.max() + 1]] = True
    return passed


assert fdr([0.001, 0.02, 0.03, 0.5]).tolist() == [True, True, True, False]  # 0.03 <= 3/4 * 0.05
assert fdr([0.04, 0.5]).tolist() == [False, False]                           # 0.04 > 1/2 * 0.05


def gistar(units, key, value, threshold):
    """Gi* z of `value` per unit (Sedona); NaN for units with no neighbour inside the band."""
    df = spark.createDataFrame(units[[key, "x", "y", value]].rename(columns={value: "val"})).selectExpr(
        f"CAST({key} AS INT) AS uid", "CAST(val AS DOUBLE) AS val", "ST_Point(x, y) AS geometry")
    w = add_binary_distance_band_column(df, threshold, include_zero_distance_neighbors=True, include_self=True,
                                        saved_attributes=["uid", "val"])
    r = g_local(w, "val", "weights", star=True).selectExpr("uid", "Z", "size(weights) AS n_neighbours").toPandas()
    r.loc[r["n_neighbours"] <= 1, "Z"] = np.nan
    return units[[key]].merge(r.rename(columns={"uid": key}), on=key, how="left")


def hotspots(units, key, levels, flag, threshold, names):
    out = []
    for lv in levels:
        ok = units[units[f"children_{lv}"] >= MIN_CHILDREN]
        if flag:
            ok = ok[~ok["state"].isin(flag[lv])]
        for var, bad in [(f"beyond_norm_%_{lv}", "hot"), (f"a_main_{lv}", "cold")]:
            r = gistar(ok, key, var, threshold).merge(ok[[key, *names, var]], on=key)
            r["p"] = 2 * norm.sf(r["Z"].abs())
            sig = r["p"].notna()
            r["significant_fdr5"] = False
            r.loc[sig, "significant_fdr5"] = fdr(r.loc[sig, "p"])
            r["spot"] = np.where(~r["significant_fdr5"], "not significant", np.where(r["Z"] > 0, "hot", "cold"))
            r["poor_access_cluster"] = r["spot"].eq(bad)  # hot spot of children far away, or cold spot of places
            out.append(r.rename(columns={var: "value"}).assign(level=lv, variable=var.replace(f"_{lv}", "")))
    return pd.concat(out, ignore_index=True)


band_india, band_kerala = band(sub), band(lb)
print(f"distance band: India sub-districts {band_india / 1000:.1f} km, Kerala local bodies {band_kerala / 1000:.1f} km")
hot_india = hotspots(sub, "subdistrict_code", LEVELS, FLAG, band_india, ["state", "district", "subdistrict"])
hot_kerala = hotspots(lb, "local_body_code", LEVELS_K, None, band_kerala, ["district", "local_body"])
for name, h in [("India sub-districts", hot_india), ("Kerala local bodies", hot_kerala)]:
    print(f"\n{name}: units in a significant poor-access cluster (FDR 5 %), and units without a neighbour")
    print(h.assign(no_neighbour=h["Z"].isna()).groupby(["level", "variable"]).agg(
        units=("Z", "size"), no_neighbour=("no_neighbour", "sum"), poor_access_cluster=("poor_access_cluster", "sum")).to_string())

distance band: India sub-districts 35.7 km, Kerala local bodies 11.0 km



India sub-districts: units in a significant poor-access cluster (FDR 5 %), and units without a neighbour
                                units  no_neighbour  poor_access_cluster
level            variable                                               
higher_secondary a_main          5899            75                  726
                 beyond_norm_%   5899            75                  810
primary          a_main          6673            71                  606
                 beyond_norm_%   6673            71                  952
secondary        a_main          6566            72                 1063
                 beyond_norm_%   6566            72                  421
upper_primary    a_main          6622            73                  833
                 beyond_norm_%   6622            73                  611

Kerala local bodies: units in a significant poor-access cluster (FDR 5 %), and units without a neighbour
                     units  no_neighbour  poor_access_clus

## Check: Sedona Gi* = the Getis–Ord formula

In [6]:
# Check: Sedona's Gi* z = the Getis-Ord formula computed directly (numpy), Kerala local bodies, every level
rows = []
for lv in LEVELS_K:
    ok = lb[lb[f"children_{lv}"] >= MIN_CHILDREN].reset_index(drop=True)
    x = ok[f"a_main_{lv}"].to_numpy()
    xy = ok[["x", "y"]].to_numpy()
    w = (np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1)) <= band_kerala).astype(float)  # self included
    n, xbar = len(x), x.mean()
    s = np.sqrt((x ** 2).mean() - xbar ** 2)
    wi, s1 = w.sum(1), (w ** 2).sum(1)
    z = (w @ x - xbar * wi) / (s * np.sqrt((n * s1 - wi ** 2) / (n - 1)))
    got = hot_kerala.query("level == @lv and variable == 'a_main'").set_index("local_body_code").loc[ok["local_body_code"], "Z"].to_numpy()
    rows.append({"level": lv, "units": n, "max_abs_diff": np.nanmax(np.abs(got - z))})
gi_check = pd.DataFrame(rows)
print(gi_check.to_string(index=False))
assert (gi_check["max_abs_diff"] < 1e-6).all(), "Sedona Gi* differs from the formula"

level  units  max_abs_diff
   lp   1024  2.664535e-14
   up   1024  1.332268e-14
   hs   1026  8.881784e-15
  hss   1022  3.375078e-14


## 2. K-Means area types

In [7]:
# K-Means area types (Spark MLlib) on India sub-districts: per level, % of children beyond the norm, log(1 + mean
# distance to the nearest school, km), log(1 + places per child); standardised. Only sub-districts with every level
# valid (no UDISE-incomplete flag, >= 100 children). K = 3..8 chosen by silhouette.
feats = [c for lv in LEVELS for c in (f"beyond_norm_%_{lv}", f"log_near_{lv}", f"log_a_{lv}")]
km = sub.copy()
for lv in LEVELS:
    km[f"log_near_{lv}"], km[f"log_a_{lv}"] = np.log1p(km[f"mean_nearest_km_{lv}"]), np.log1p(km[f"a_main_{lv}"])
valid = np.ones(len(km), bool)
for lv in LEVELS:
    valid &= (km[f"children_{lv}"] >= MIN_CHILDREN).to_numpy() & ~km["state"].isin(FLAG[lv]).to_numpy()
left_out = km[~valid]
km = km[valid].reset_index(drop=True)
assert km[feats].notna().all().all()
print("sub-districts clustered:", len(km), "| left out:", len(left_out), left_out["state"].value_counts().head(6).to_dict())
kdf = spark.createDataFrame(km[["subdistrict_code", *feats]].rename(columns=lambda c: c.replace("%", "pct")))
fcols = [f.replace("%", "pct") for f in feats]
raw = VectorAssembler(inputCols=fcols, outputCol="raw").transform(kdf)
kdf = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True).fit(raw).transform(raw).cache()
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
sil = []
for k in range(3, 9):
    pred = KMeans(k=k, seed=1, maxIter=100, featuresCol="features", predictionCol="cluster").fit(kdf).transform(kdf)
    sil.append({"k": k, "silhouette": evaluator.evaluate(pred)})
sil = pd.DataFrame(sil)
best_k = int(sil.loc[sil["silhouette"].idxmax(), "k"])
print(sil.round(4).to_string(index=False), "\nchosen K:", best_k)
labels = {}
for seed in (1, 2):
    labels[seed] = KMeans(k=best_k, seed=seed, maxIter=100, featuresCol="features", predictionCol="cluster").fit(kdf).transform(kdf) \
        .select("subdistrict_code", "cluster").toPandas().set_index("subdistrict_code")["cluster"]
# stability: share of sub-districts in the same cluster under another seed (clusters matched one-to-one)
ct = pd.crosstab(labels[1], labels[2].loc[labels[1].index])
r, c = linear_sum_assignment(-ct.to_numpy())
stability = ct.to_numpy()[r, c].sum() / len(labels[1])
print(f"same cluster with seed 2: {stability:.1%}")
km["cluster"] = km["subdistrict_code"].map(labels[1])
prof = []
for k, g in km.groupby("cluster"):
    row = {"cluster": k, "subdistricts": len(g), "children_primary_M": g["children_primary"].sum() / 1e6,
           "top_states": ", ".join(g["state"].value_counts().head(3).index)}
    for lv in LEVELS:
        w = g[f"children_{lv}"]
        for m in ("beyond_norm_%", "mean_nearest_km", "a_main"):
            row[f"{m}_{lv}"] = np.average(g[f"{m}_{lv}"], weights=w)
    prof.append(row)
profiles = pd.DataFrame(prof)
print(profiles.round(2).to_string(index=False))

sub-districts clustered: 5890 | left out: 920 {'Andhra Pradesh': 666, 'Arunāchal Pradesh': 95, 'Nāgāland': 68, 'Delhi': 33, 'Jammu and Kashmīr': 19, 'Manipur': 16}


 k  silhouette
 3      0.3900
 4      0.3929
 5      0.4153
 6      0.3459
 7      0.3490
 8      0.3438 
chosen K: 5


same cluster with seed 2: 81.0%
 cluster  subdistricts  children_primary_M                                     top_states  beyond_norm_%_primary  mean_nearest_km_primary  a_main_primary  beyond_norm_%_upper_primary  mean_nearest_km_upper_primary  a_main_upper_primary  beyond_norm_%_secondary  mean_nearest_km_secondary  a_main_secondary  beyond_norm_%_higher_secondary  mean_nearest_km_higher_secondary  a_main_higher_secondary
       0          1882               10.11               Telangāna, Tamil Nādu, Rājasthān                  13.36                     0.55            2.10                         1.05                           0.83                  2.13                     0.56                       1.29              1.52                            1.71                              2.04                     0.97
       1            60                0.52 Arunāchal Pradesh, Madhya Pradesh, West Bengal                  84.00                     5.53            0.32                     

## 3. FP-Growth: problems that occur together

In [8]:
# FP-Growth (Spark MLlib): which access problems occur together in a sub-district. Items per valid level:
# far_<level> = % of children beyond the norm in the worst quarter of sub-districts (and > 0);
# few_places_<level> = places per child in the lowest quarter.
fp = sub.copy()
items = [[] for _ in range(len(fp))]
for lv in LEVELS:
    ok = (fp[f"children_{lv}"] >= MIN_CHILDREN) & ~fp["state"].isin(FLAG[lv])
    far_cut, few_cut = fp.loc[ok, f"beyond_norm_%_{lv}"].quantile(0.75), fp.loc[ok, f"a_main_{lv}"].quantile(0.25)
    far = ok & (fp[f"beyond_norm_%_{lv}"] >= far_cut) & (fp[f"beyond_norm_%_{lv}"] > 0)
    few = ok & (fp[f"a_main_{lv}"] <= few_cut)
    print(f"{lv}: far = beyond norm >= {far_cut:.1f} %, few places = places per child <= {few_cut:.3f}")
    for i in np.nonzero(far.to_numpy())[0]:
        items[i].append(f"far_{lv}")
    for i in np.nonzero(few.to_numpy())[0]:
        items[i].append(f"few_places_{lv}")
fp["items"] = items
model = FPGrowth(itemsCol="items", minSupport=0.05, minConfidence=0.6).fit(
    spark.createDataFrame(fp[["subdistrict_code", "items"]]))
rules = model.associationRules.toPandas()
rules["antecedent"], rules["consequent"] = rules["antecedent"].map(sorted), rules["consequent"].map(sorted)
rules = rules.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)
print("transactions:", len(fp), "| rules:", len(rules))
print(rules.head(15).assign(antecedent=lambda d: d["antecedent"].map(" + ".join), consequent=lambda d: d["consequent"].map(" + ".join))
      .round(3).to_string(index=False))
# check: recount support and confidence of every rule in pandas
sets = fp["items"].map(set)
for r in rules.itertuples():
    a, both = sets.map(set(r.antecedent).issubset), sets.map(set(r.antecedent + r.consequent).issubset)
    assert abs(both.mean() - r.support) < 1e-12 and abs(both.sum() / a.sum() - r.confidence) < 1e-12, r
print("support and confidence recounted for all rules (check passed)")

primary: far = beyond norm >= 25.9 %, few places = places per child <= 0.946
upper_primary: far = beyond norm >= 3.7 %, few places = places per child <= 0.829
secondary: far = beyond norm >= 3.4 %, few places = places per child <= 0.495
higher_secondary: far = beyond norm >= 14.4 %, few places = places per child <= 0.185


transactions: 6810 | rules: 36
                                                                   antecedent               consequent  confidence  lift  support
  few_places_higher_secondary + few_places_primary + few_places_upper_primary     few_places_secondary       0.959 3.977    0.103
                       few_places_higher_secondary + few_places_upper_primary     few_places_secondary       0.956 3.964    0.114
      few_places_higher_secondary + few_places_primary + few_places_secondary few_places_upper_primary       0.958 3.938    0.103
                                    few_places_primary + few_places_secondary few_places_upper_primary       0.941 3.872    0.156
                             few_places_higher_secondary + few_places_primary     few_places_secondary       0.934 3.872    0.107
                             few_places_higher_secondary + few_places_primary few_places_upper_primary       0.932 3.834    0.107
few_places_higher_secondary + few_places_secondary + few_pl

## 4. Kerala: clusters of underserved cells

In [9]:
# Kerala: clusters of underserved cells per level = connected groups of 100 m cells with no school of the level within
# the norm (8-neighbour; the same as DBSCAN with eps 150 m and minPts 1 on this grid, exact). Input for D7b.
width, _, tr = pc.grid(PROJECT / "data" / "raw" / "worldpop")
k = pd.read_parquet(GOLD / "school_access_kerala", columns=["cell_id", "state", "district_code", "local_body_code", "pop",
                                                         *[f"{p}_{lv}" for lv in LEVELS_K for p in ("dem", "schools_in_reach")]])
fixd = pd.read_parquet(SILVER / "cell_units_fixed", columns=["cell_id", "district_code"]).set_index("cell_id")["district_code"]
k["district_code"] = k["cell_id"].map(fixd).fillna(k["district_code"]).astype(int)
k["row"], k["col"] = k["cell_id"] // width, k["cell_id"] % width
r0, c0 = k["row"].min(), k["col"].min()
names_d = pc.load_units(SILVER, "district_code").set_index("code")["unit_name"]
names_lb = pc.load_units(SILVER, "local_body_code").set_index("code")["unit_name"]
dK = pd.read_csv(GOLD / "school_kerala_by_district.csv")
clusters, ccheck = [], []
for lv in LEVELS_K:
    far = k[k[f"schools_in_reach_{lv}"].isna()]
    grid = np.zeros((k["row"].max() - r0 + 1, k["col"].max() - c0 + 1), bool)
    grid[far["row"] - r0, far["col"] - c0] = True
    lab, n = ndimage.label(grid, structure=np.ones((3, 3)))
    far = far.assign(cluster=lab[far["row"] - r0, far["col"] - c0], w=far[f"dem_{lv}"])
    expected = (dK[f"beyond_norm_%_{lv}"] / 100 * dK[f"children_{lv}"]).sum()  # D6 beyond-norm children (Kerala + Mahe)
    ccheck.append({"level": lv, "clusters": n, "children_in_clusters": far["w"].sum(), "d6_beyond_norm_children": expected})
    far = far.assign(wc=far["w"] * (far["col"] + 0.5), wr=far["w"] * (far["row"] + 0.5))
    g = far.groupby("cluster")
    c = pd.DataFrame({"cells": g.size(), "children": g["w"].sum(), "people": g["pop"].sum(), "wc": g["wc"].sum(), "wr": g["wr"].sum()})
    c = c[c["children"] >= 50].copy()
    c["lon"] = tr.c + c["wc"] / c["children"] * tr.a  # children-weighted centre of the cluster
    c["lat"] = tr.f + c["wr"] / c["children"] * tr.e
    for u in ["district_code", "local_body_code"]:  # the unit holding most of the cluster's children
        m = far[far["cluster"].isin(c.index)].groupby(["cluster", u])["w"].sum().reset_index()
        c[u] = m.sort_values("w").groupby("cluster").tail(1).set_index("cluster")[u]
    c = c.drop(columns=["wc", "wr"]).sort_values("children", ascending=False).reset_index()
    c.insert(0, "level", lv)
    c["district"] = c["district_code"].map(names_d)
    c["local_body"] = c["local_body_code"].map(names_lb).fillna("(none)")
    clusters.append(c)
ccheck = pd.DataFrame(ccheck)
print(ccheck.round(1).to_string(index=False))
assert np.allclose(ccheck["children_in_clusters"], ccheck["d6_beyond_norm_children"], rtol=1e-6), "cluster children != D6"
clusters = pd.concat(clusters, ignore_index=True)
print("\nclusters with >= 50 children:", clusters.groupby("level").agg(clusters=("children", "size"), children=("children", "sum")).round(0).to_dict("index"))
print(clusters.groupby("level").head(5)[["level", "district", "local_body", "cells", "children", "lon", "lat"]].round(4).to_string(index=False))
del k

level  clusters  children_in_clusters  d6_beyond_norm_children
   lp     10141              191916.6                 191916.6
   up      3708                9572.0                   9572.0
   hs      2573                5432.8                   5432.8
  hss      1248                1093.7                   1093.7

clusters with >= 50 children: {'hs': {'clusters': 19, 'children': 2021.0}, 'hss': {'clusters': 3, 'children': 274.0}, 'lp': {'clusters': 360, 'children': 170844.0}, 'up': {'clusters': 29, 'children': 5356.0}}
level       district            local_body  cells   children     lon     lat
   lp      Ernakulam Vadavukode-Puthencruz 100740 26147.2365 76.5255  9.9548
   lp       Thrissur Chalakudy Muncipality  49559 17391.2338 76.2938 10.4245
   lp      Kasaragod         Pullur-Periya  90617 10144.2058 75.3202 12.2682
   lp       Palakkad             Pudussery  49472 10128.3898 76.7354 10.7085
   lp        Wayanad          Mananthavady  64643  7990.5646 76.0999 11.7077
   up       P

## Save

In [10]:
RESULTS.mkdir(parents=True, exist_ok=True)
out = {"school_hotspots_india_subdistrict.csv": hot_india, "school_hotspots_kerala_local_body.csv": hot_kerala,
       "school_gistar_check.csv": gi_check, "school_kmeans_silhouette.csv": sil,
       "school_area_types_india_subdistrict.csv": km[["state", "district", "subdistrict_code", "subdistrict", "cluster"]],
       "school_area_type_profiles.csv": profiles, "school_fp_rules.csv": rules.assign(
           antecedent=rules["antecedent"].map(" + ".join), consequent=rules["consequent"].map(" + ".join)),
       "school_underserved_clusters_kerala.csv": clusters}
for name, t in out.items():
    t.to_csv(GOLD / name, index=False)
    shutil.copy2(GOLD / name, RESULTS / name)
print("written:", list(out))
spark.stop()

written: ['school_hotspots_india_subdistrict.csv', 'school_hotspots_kerala_local_body.csv', 'school_gistar_check.csv', 'school_kmeans_silhouette.csv', 'school_area_types_india_subdistrict.csv', 'school_area_type_profiles.csv', 'school_fp_rules.csv', 'school_underserved_clusters_kerala.csv']
